In [20]:
import time
import logging
from datetime import datetime
import pandas as pd

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s"
)


class TradingMonitor:
    """
    Continuously monitors the Bitcoin market
    and sends observations to TradingAgent.

    PAPER TRADING ONLY.
    """

    def __init__(self, agent, interval_minutes=30):

        self.agent = agent
        self.interval_seconds = interval_minutes * 60
        self.running = False

    # =========================================================
    # GET MARKET DATA
    # =========================================================

    def get_market_observation(self):

        df = pd.read_csv('../data/btc_indicators.csv')

        if df.empty:
            raise ValueError(
                "No BTC market data received."
            )

        latest = df.iloc[-1]

        return {
            "price": float(latest["close"]),
            "atr": float(latest["atr_14"]),
            "rsi": float(latest["rsi_14"]),
            "macd": float(latest["macd"]),
            "macd_signal": float(
                latest["macd_signal"]
            ),
            "volume_ratio": float(
                latest["volume_ratio"]
            )
        }

    # =========================================================
    # RUN ONCE
    # =========================================================

    def run_once(self):

        logging.info(
            "Running market analysis..."
        )

        market = self.get_market_observation()

        logging.info(
            f"BTC price: ${market['price']:,.2f}"
        )

        result = self.agent.process_market_data(
            price=market["price"],
            atr=market["atr"],
            rsi=market["rsi"],
            macd=market["macd"],
            macd_signal=market["macd_signal"],
            volume_ratio=market["volume_ratio"]
        )

        logging.info(
            f"Hybrid decision: "
            f"{result.get('hybrid_recommendation')}"
        )

        logging.info(
            f"Trade: {result.get('trade')}"
        )

        return result

    # =========================================================
    # CONTINUOUS MONITORING
    # =========================================================

    def start(self):

        self.running = True

        logging.info(
            "Bitcoin Trading Monitor started."
        )

        logging.info(
            f"Monitoring interval: "
            f"{self.interval_seconds // 60} minutes"
        )

        while self.running:

            try:

                self.run_once()

            except Exception as exc:

                logging.exception(
                    f"Monitoring error: {exc}"
                )

            logging.info(
                "Waiting for next monitoring cycle..."
            )

            time.sleep(
                self.interval_seconds
            )

    # =========================================================
    # STOP
    # =========================================================

    def stop(self):

        self.running = False

        logging.info(
            "Bitcoin Trading Monitor stopped."
        )

In [21]:
import pandas as pd
df = pd.read_csv('../data/btc_indicators.csv')


In [22]:
from config_loader import (
    get_google_sheet_config,
    convert_config_values
)

rows = get_google_sheet_config()

config = convert_config_values(rows)

interval = int(
    config.get(
        "monitoring_interval_minutes",
        30
    )
)

from trading_agent import TradingAgent

agent = TradingAgent(config)

monitor = TradingMonitor(
    agent=agent,
    interval_minutes=interval
)

2026-08-31 18:19:05,094 | INFO | file_cache is only supported with oauth2client<4.0.0


In [23]:
result = monitor.run_once()

print(result)

2026-08-31 18:19:07,382 | INFO | Running market analysis...
2026-08-31 18:19:07,393 | INFO | BTC price: $77,524.59
2026-08-31 18:19:18,867 | INFO | HTTP Request: POST https://api.openai.com/v1/responses "HTTP/1.1 200 OK"
2026-08-31 18:19:19,406 | INFO | Hybrid decision: AUTO
2026-08-31 18:19:19,406 | INFO | Trade: {'timestamp': '2026-08-31T18:19:18.974697', 'action': 'BUY', 'strategy': 'DCA', 'btc_price': 77524.59, 'btc_quantity': 0.006449566518184746, 'usd_amount': 500.0, 'fee': 0.75, 'reason': 'Initial DCA purchase'}


DCA BUY executed: $500.00 at $77,524.59
{'timestamp': '2026-08-31T18:19:07.394601', 'price': 77524.59, 'dca': {'action': 'BUY', 'amount_usd': 500.0, 'price': 77524.59, 'reason': 'Initial DCA purchase'}, 'atr': {'action': 'HOLD', 'price': 77524.59, 'atr': 352.51520307606495, 'confirmations': 0, 'reason': 'Insufficient bullish confirmations'}, 'trade': {'timestamp': '2026-08-31T18:19:18.974697', 'action': 'BUY', 'strategy': 'DCA', 'btc_price': 77524.59, 'btc_quantity': 0.006449566518184746, 'usd_amount': 500.0, 'fee': 0.75, 'reason': 'Initial DCA purchase'}, 'llm': {'market_regime': 'bearish', 'recommendation': 'DCA', 'confidence': 0.65, 'reason': 'MACD is negative and below its signal line indicating bearish momentum; RSI ~36 is mildly oversold but not signaling a clear reversal. Volume ratio is low, suggesting weak participation and limited conviction for a bounce. ATR is small relative to price (low volatility), so this looks like a down/quiet market rather than a high-volatility regi